# BioForge Fase 2: Diseño de Binders para Alfa-sinucleína (Dominio NAC)

Este notebook automatiza el diseño de binders computacionales usando **RFdiffusion** y **ProteinMPNN**. El objetivo es bloquear el dominio NAC (residuos 61-95) de la Alfa-sinucleína (P37840) para prevenir su agregación.

### 1. Verificación de GPU e Instalación
Instalamos RFdiffusion y descargamos los pesos del modelo.

In [ ]:
import torch
if not torch.cuda.is_available():
  print("ERROR: No se detectó GPU. Ve a Runtime -> Change runtime type -> T4 GPU.")
else:
  print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

!pip install -q py3Dmol biopython icecream

import os
if not os.path.isdir("RFdiffusion"):
  print("Clonando RFdiffusion...")
  !git clone https://github.com/RosettaCommons/RFdiffusion.git
  !pip install -q dgl==1.0.2 -f https://data.dgl.ai/wheels/cu117/repo.html
  !pip install -q -e ./RFdiffusion
  !mkdir -p RFdiffusion/weights
  !wget -q http://files.ipd.uw.edu/pub/rf_diffusion/6f5902ac2370aa4dda0a36be5a86ed8b/Base_weights.pt -P RFdiffusion/weights/

!mkdir -p outputs

### 2. Carga de Estructura Target (NAC Domain)
Cargamos el dominio NAC extraído de BioForge.

In [ ]:
repo_url = "https://raw.githubusercontent.com/DSANQUIZR/BioForge/main/inputs/P37840_NAC.pdb"
!wget -O target.pdb {repo_url}

import py3Dmol
viewer = py3Dmol.view(query='target.pdb', format='pdb')
viewer.setStyle({'cartoon': {'color': 'spectrum'}})
viewer.show()

### 3. Ejecución de RFdiffusion
Diseñamos un binder de 50 residuos que se una al hotspot del dominio NAC.

In [ ]:
HOTSPOT = "A61-95"
BINDER_LEN = "50"

!python RFdiffusion/scripts/run_inference.py \
  inference.output_prefix=outputs/design \
  inference.model_directory_path=RFdiffusion/weights \
  inference.input_pdb=target.pdb \
  'contig.contig_conf_list=["A1-35/0 50-50"]' \
  'inference.num_designs=2'

### 4. Secuenciación con ProteinMPNN
Generamos las secuencias para los diseños.

In [ ]:
if not os.path.isdir("ProteinMPNN"):
  !git clone https://github.com/dauparas/ProteinMPNN.git

import glob
design_files = glob.glob("outputs/design_*.pdb")

for i, pdb in enumerate(design_files):
  print(f"Secuenciando {pdb}...")
  !python ProteinMPNN/protein_mpnn_run.py \
    --pdb_path {pdb} \
    --out_folder outputs/mpnn_results \
    --num_seq_per_target 2

### 5. Empaquetado de Resultados
Creamos un ZIP con todos los binders para descarga automática.

In [ ]:
!zip -r binders_result.zip outputs/
print("### SUCCESS: BINDERS GENERATED AND ZIPPED ###")